# Table 2 — Classical LSTM vs Augmented LSTM

**Paper:** Fecamp, Mikael, Warin (2019/2020) — *Deep learning for discrete-time hedging in incomplete markets*, arXiv:1902.05287v4, §4.3.1 Table 2 (p. 8).

### What this notebook reproduces

A 2x2 MSE table comparing two recurrent architectures on two payoffs:

|                    | Black-Scholes call | 2-markets spread |
|--------------------|--------------------|------------------|
| Classical LSTM     | 5.73e-05           | 3.64e-04         |
| Augmented LSTM     | 3.97e-05           | 1.11e-04         |

Loss = MSE of Y = pnl - g(S_T) (Eq. 4). **No** liquidity constraint, **no** transaction costs — this is the frictionless setup.

### Design: `Config` + `run_experiment`

Every "thing that might change" between experiments (dataset, payoff, loss, architectures, constraints) is a field of `Config`. The runner `run_experiment(cfg)` trains every model in `cfg.arch_factory` on one shared set of paths and returns a uniform results dict.

The extension notebooks (`03_extension_dataset.ipynb`, `04_extension_modelling.ipynb`) import the **same runner** and only override one or two config fields — so differences vs this baseline are a clean config-diff.

### Paper parameters (Table 2 caption)

* Call option: `T = 3/12`, `dt = 1/360` -> `N = 90`; `S0 = K = 1`, `sigma = 0.3`, `mu = 0.02`.
* Spread option: same `T, dt, N`; `S0 = (1, 0.5)`, `K = 0.5`, `sigma = (0.3, 0.3)`, `mu = (0.02, 0.02)`, `corr(W^1, W^2) = 0.2`. Payoff = `(S_T^1 - S_T^2 - K)^+`.
* Training (paper §4.3): Adam `lr = 1e-3`, batch 50, 20 000 iterations, batch-norm stats computed on 100 000 reference paths.
* Networks (paper §4.3): LSTM cell 50 hidden units, FF head = 3 ReLU layers of width 10.


## 0 - Imports


In [ ]:
import sys, os, json
# Make `src` importable regardless of where the notebook is launched from
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.simulators    import simulate_gbm, simulate_multi_gbm, compute_norm_stats
from src.payoffs       import call_payoff, spread_payoff
from src.architectures import ClassicalLSTM, AugmentedLSTM
from src.losses        import mse
from src.experiment    import Config, run_experiment, report_table

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Device:', DEVICE)


## 1 - Paper parameters (Table 2 caption)


In [ ]:
# Horizon & grid (shared by both sub-experiments)
T  = 3/12
DT = 1/360
N  = int(round(T / DT))          # should be 90
assert N == 90, f"Expected N=90 per paper Table 2 (T=3/12, dt=1/360), got {N}"

# Training (paper §4.3 defaults — also live in Config as defaults)
BATCH_SIZE = 50
LR         = 1e-3
N_ITER     = 20_000
N_NORM     = 100_000             # reference pool for batch-norm stats
N_TRAIN    = 50_000              # same size as Table 1 notebook
N_EVAL     = 100_000

# Architecture (paper §4.3)
LSTM_HIDDEN = 50
FF_WIDTH    = 10
FF_LAYERS   = 3

print(f'Hedging steps N = {N}  (T = {T:.4f} yr, dt = {DT:.6f})')
print(f'Pool sizes:  train={N_TRAIN:,}   eval={N_EVAL:,}   ref(norm)={N_NORM:,}')


## 2 - Dataset factories

Each `paths_factory` returns `(train_raw, eval_raw, norm_mean, norm_std)`. Swapping a factory is the entire dataset-swap surface area — this is **Person B's seam**.

Batch-norm stats (paper §4.3) are computed on a **separate** 100 000-path reference pool, not on the training set, so stats are independent of training/eval randomness.


In [ ]:
def make_bs_call_paths():
    """Single-asset GBM for the vanilla call column of Table 2.

    Paper caption: T=3/12, dt=1/360, S0=K=1, sigma=0.3, mu=0.02.
    """
    S0, sigma, mu = 1.0, 0.3, 0.02
    train = simulate_gbm(N_TRAIN, S0, mu, sigma, DT, N, device=DEVICE, seed=0)
    evl   = simulate_gbm(N_EVAL,  S0, mu, sigma, DT, N, device=DEVICE, seed=1)
    ref   = simulate_gbm(N_NORM,  S0, mu, sigma, DT, N, device=DEVICE, seed=99)
    mean, std = compute_norm_stats(ref)
    return train, evl, mean, std


def make_spread_paths():
    """Two correlated GBMs for the spread column of Table 2.

    Paper caption: S0=(1, 0.5), sigma=(0.3, 0.3), mu=(0.02, 0.02),
                   corr(W^1, W^2) = 0.2.
    """
    S0    = np.array([1.0, 0.5])
    mu    = np.array([0.02, 0.02])
    sigma = np.array([0.3, 0.3])
    corr  = np.array([[1.0, 0.2], [0.2, 1.0]])
    train = simulate_multi_gbm(N_TRAIN, S0, mu, sigma, corr, DT, N, device=DEVICE, seed=10)
    evl   = simulate_multi_gbm(N_EVAL,  S0, mu, sigma, corr, DT, N, device=DEVICE, seed=11)
    ref   = simulate_multi_gbm(N_NORM,  S0, mu, sigma, corr, DT, N, device=DEVICE, seed=109)
    mean, std = compute_norm_stats(ref)
    return train, evl, mean, std

# Sanity-check shapes (no training yet)
_tr, _ev, _m, _s = make_bs_call_paths()
print('BS call   :', _tr.shape, '  mean/std shape:', _m.shape, _s.shape)
del _tr, _ev, _m, _s
_tr, _ev, _m, _s = make_spread_paths()
print('2-mkt spread:', _tr.shape, '  mean/std shape:', _m.shape, _s.shape)
del _tr, _ev, _m, _s


## 3 - Experiment configs

Two configs, one per sub-experiment. Both use the same MSE loss, the same 20 000-iter Adam schedule, and `liq = inf`, `tc_cost = None` (all paper defaults, which is why we don't pass them explicitly below — the `Config` dataclass carries the paper defaults).

`paper_values` is consumed by `report_table` to print the ratio vs paper. Extension configs can set it to `{}` to skip that column.


In [ ]:
cfg_bs = Config(
    name          = 'BS call',
    d             = 1,
    N             = N,
    paths_factory = make_bs_call_paths,
    # BS call payoff — single asset, strike K = 1.0. Paths are (batch, N+1);
    # call_payoff works on a 1-D S_T vector.
    payoff_fn     = lambda p: call_payoff(p[..., -1, 0] if p.dim()==3 else p[:, -1], K=1.0),
    arch_factory  = {
        'Classical LSTM': lambda: ClassicalLSTM(
            N=N, d=1, hidden=LSTM_HIDDEN),
        'Augmented LSTM': lambda: AugmentedLSTM(
            N=N, d=1, hidden=LSTM_HIDDEN, ff_width=FF_WIDTH, ff_layers=FF_LAYERS),
    },
    n_iter        = N_ITER,
    batch_size    = BATCH_SIZE,
    lr            = LR,
    paper_values  = {'Classical LSTM': 5.73e-05, 'Augmented LSTM': 3.97e-05},
)

cfg_spread = Config(
    name          = '2-markets spread',
    d             = 2,
    N             = N,
    paths_factory = make_spread_paths,
    # Spread payoff (S_T^1 - S_T^2 - K)^+ with K = 0.5. Paths are (batch, N+1, 2).
    payoff_fn     = lambda p: spread_payoff(p[:, -1, :], K=0.5),
    arch_factory  = {
        'Classical LSTM': lambda: ClassicalLSTM(
            N=N, d=2, hidden=LSTM_HIDDEN),
        'Augmented LSTM': lambda: AugmentedLSTM(
            N=N, d=2, hidden=LSTM_HIDDEN, ff_width=FF_WIDTH, ff_layers=FF_LAYERS),
    },
    n_iter        = N_ITER,
    batch_size    = BATCH_SIZE,
    lr            = LR,
    paper_values  = {'Classical LSTM': 3.64e-04, 'Augmented LSTM': 1.11e-04},
)

# A tiny summary so we can eyeball the configs before a 2-hour training run
for cfg in (cfg_bs, cfg_spread):
    print(f'{cfg.name:<20}  d={cfg.d}  N={cfg.N}  '
          f'archs={list(cfg.arch_factory)}  paper={cfg.paper_values}')


## 4 - Train both experiments

Each experiment trains 2 models -> 4 total. With N=90 (Table 2) vs N=30 (Table 1) each iteration is ~3x slower. Budget accordingly — ~overnight for all four at 20k iters on CPU, ~30-60 min on a single GPU.


In [ ]:
results = {}
results[cfg_bs.name]     = run_experiment(cfg_bs,     device=DEVICE)
results[cfg_spread.name] = run_experiment(cfg_spread, device=DEVICE)

for exp_name, res in results.items():
    print(f'\n--- {exp_name} ---')
    for name, m in res['mse'].items():
        paper = res['cfg'].paper_values.get(name, None)
        print(f'  {name:<20}  our MSE = {m:.3e}   paper = {paper}')


## 5 - Table 2 reproduction


In [ ]:
report_table(results, title=f'TABLE 2 REPRODUCTION  —  MSE,  N={N}, T={T:.4f}yr')


## 6 - Diagnostics: loss curves

Log-scale MSE on train (per-iter, heavily smoothed by the eye) and on the 100 000-path eval set (checkpoints every 1 000 iters).


In [ ]:
fig, axes = plt.subplots(len(results), 2, figsize=(11, 3.6 * len(results)), squeeze=False)
for r, (exp_name, res) in enumerate(results.items()):
    for c, (name, h) in enumerate(res['histories'].items()):
        ax = axes[r, c]
        # Train loss is noisy at batch=50; take a running mean for visibility.
        tr = np.array(h['train'])
        win = max(1, len(tr) // 200)
        tr_smooth = np.convolve(tr, np.ones(win)/win, mode='valid')
        ax.plot(np.arange(len(tr_smooth)), tr_smooth, lw=0.8, alpha=0.6, label=f'train (smooth w={win})')
        ax.plot(h['test_iters'], h['test'], 'o-', color='C1', label='test MSE')
        paper = res['cfg'].paper_values.get(name, None)
        if paper is not None:
            ax.axhline(paper, color='red', ls='--', lw=0.8, label=f'paper = {paper:.2e}')
        ax.set(title=f'{exp_name}  —  {name}', xlabel='iteration', ylabel='MSE', yscale='log')
        ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 7 - Diagnostics: hedging-error distributions

Y = pnl - g(S_T) (tc = 0 here since no transaction costs). A tighter / more symmetric distribution around zero means a better hedge.


In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 4), squeeze=False)
for c, (exp_name, res) in enumerate(results.items()):
    ax = axes[0, c]
    for name, pnl in res['pnl'].items():
        ax.hist(pnl, bins=80, alpha=0.45, label=f'{name}  (sd={pnl.std():.3e})', density=True)
    ax.axvline(0, color='k', ls='--', lw=0.7)
    ax.set(title=f'Hedging error Y  —  {exp_name}', xlabel='Y', ylabel='density')
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


## 8 - Save results

Robust save pattern (same as Table 1): histories go through a `_clean` pass so tensors / numpy scalars don't break `json.dump`, and each JSON write is size-reported so a silent zero-byte write is visible.


In [ ]:
OUT = os.path.join(ROOT, 'results_table2')
os.makedirs(OUT, exist_ok=True)


def _clean(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, (list, tuple)):
        return [_clean(v) for v in x]
    if isinstance(x, dict):
        return {k: _clean(v) for k, v in x.items()}
    return x


def _slug(s: str) -> str:
    return s.lower().replace(' ', '_').replace('-', '_')


# --- weights, histories, pnls (per experiment) ---
for exp_name, res in results.items():
    exp_slug = _slug(exp_name)
    for model_name, model in res['models'].items():
        tag = _slug(model_name)
        torch.save(model.state_dict(), os.path.join(OUT, f'{exp_slug}__{tag}.pt'))
    # strip best_state (duplicates .pt and isn't JSON-serialisable)
    hist = {
        m: {k: _clean(v) for k, v in h.items() if k != 'best_state'}
        for m, h in res['histories'].items()
    }
    hp = os.path.join(OUT, f'{exp_slug}__histories.json')
    with open(hp, 'w') as f:
        json.dump(hist, f, indent=2, default=_clean)
    print(f'  histories: {hp}  ({os.path.getsize(hp):,} bytes)')
    for model_name, pnl in res['pnl'].items():
        tag = _slug(model_name)
        np.save(os.path.join(OUT, f'{exp_slug}__pnl_{tag}.npy'), pnl)

# --- one combined summary JSON ---
summary = {
    'params': {
        'T': T, 'dt': DT, 'N': N,
        'batch_size': BATCH_SIZE, 'lr': LR, 'n_iter': N_ITER,
        'lstm_hidden': LSTM_HIDDEN, 'ff_width': FF_WIDTH, 'ff_layers': FF_LAYERS,
        'n_train': N_TRAIN, 'n_eval': N_EVAL, 'n_norm': N_NORM,
    },
    'results': {
        exp_name: {
            model_name: {
                'our_mse'  : res['mse'][model_name],
                'paper_mse': res['cfg'].paper_values.get(model_name),
            }
            for model_name in res['mse']
        }
        for exp_name, res in results.items()
    },
}
sp = os.path.join(OUT, 'table2_summary.json')
with open(sp, 'w') as f:
    json.dump(summary, f, indent=2, default=_clean)
print(f'\nSummary: {sp}  ({os.path.getsize(sp):,} bytes)')

print('\nSaved to', OUT)
for fn in sorted(os.listdir(OUT)):
    print('  ', fn)


---

### How the extension notebooks reuse this

Person B's dataset swap:

```python
def make_spx_paths():
    ...  # load real SPX returns, bootstrap paths, return (train, eval, mean, std)
    return train, evl, mean, std

cfg_spx = replace(cfg_bs, name='SPX call', paths_factory=make_spx_paths, paper_values={})
res_spx = run_experiment(cfg_spx, device=DEVICE)
```

Person C's risk-measure swap:

```python
from src.losses import moment_2_4  # or a new CVaR loss
cfg_cvar = replace(cfg_bs, name='BS call / CVaR', loss_fn=cvar_loss, paper_values={})
```

Everything else — the training loop, the PnL computation, the reporter — is identical, so the comparison vs the Table 2 baseline is a clean diff.
